# DreamSnap — Free Training Worker (operator-run)

Keeps a GPU notebook running as a **training worker**: it polls your backend for
models waiting to be trained, trains a Flux LoRA for each, uploads the weights, and
marks the model `COMPLETED`. **You (the app operator) run this — your users never
touch Colab.** They just upload images + click *Train*; this worker does the rest.

## Setup
1. Backend in self-hosted mode: `TRAINING_MODE=manual` + `WORKER_SECRET` set on Render.
2. Runtime → **Change runtime type → GPU**.
3. Hugging Face token with access to `black-forest-labs/FLUX.1-dev`.

> Run this on a **separate** GPU session from the generation worker — a single T4
> can't hold both the training subprocess and the generation pipeline at once.
>
> Training is serial: each model takes ~1–3 h on a free T4, so a queue of users
> waits its turn. Free sessions die after ~9–12 h; if it stops, queued models stay
> `TRAINING` until you re-run this. Kaggle is more reliable than Colab.

## 1. Configure

In [ ]:
BACKEND_URL   = "https://dreamsnap.onrender.com"   # your deployed backend
WORKER_SECRET = "PASTE_WORKER_SECRET"              # must match WORKER_SECRET on the backend
HF_TOKEN      = "hf_xxxxxxxxxxxxxxxx"               # token with FLUX.1-dev access

STEPS        = 1000        # 800-1500 typical
RESOLUTION   = [512, 768]  # drop to [512] if you hit out-of-memory
POLL_SECONDS = 20          # how often to check for new training jobs

## 2. Install ai-toolkit

In [ ]:
!nvidia-smi -L
%cd /content
![ -d ai-toolkit ] || git clone https://github.com/ostris/ai-toolkit.git
%cd /content/ai-toolkit
!git submodule update --init --recursive
!pip install -q -r requirements.txt
!pip install -q requests pyyaml huggingface_hub

from huggingface_hub import login
login(token=HF_TOKEN)
print("ready")

## 3. Helpers (train one job, upload weights)

In [ ]:
import os, glob, time, shutil, subprocess, requests, yaml

HEADERS = {"x-worker-secret": WORKER_SECRET}

def prepare_dataset(job):
    d = f"/content/dataset_{job['id'][:8]}"
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d)
    trigger = (job.get("triggerWord") or "subject").strip()
    for i, url in enumerate(job["imageUrls"]):
        img = requests.get(url, timeout=120); img.raise_for_status()
        ext = url.split("?")[0].split(".")[-1].lower()
        if ext not in ("jpg", "jpeg", "png", "webp"):
            ext = "jpg"
        open(os.path.join(d, f"img_{i}.{ext}"), "wb").write(img.content)
        open(os.path.join(d, f"img_{i}.txt"), "w").write(trigger)
    return d, trigger

def write_config(job, dataset_dir, trigger):
    name = f"lora_{job['id'][:8]}"
    out_dir = "/content/output"
    os.makedirs(out_dir, exist_ok=True)
    config = {
        "job": "extension",
        "config": {"name": name, "process": [{
            "type": "sd_trainer",
            "training_folder": out_dir,
            "device": "cuda:0",
            "trigger_word": trigger,
            "network": {"type": "lora", "linear": 16, "linear_alpha": 16},
            "save": {"dtype": "float16", "save_every": STEPS, "max_step_saves_to_keep": 1},
            "datasets": [{"folder_path": dataset_dir, "caption_ext": "txt",
                          "caption_dropout_rate": 0.05, "cache_latents_to_disk": True,
                          "resolution": RESOLUTION}],
            "train": {"batch_size": 1, "steps": STEPS, "gradient_accumulation_steps": 1,
                      "train_unet": True, "train_text_encoder": False,
                      "gradient_checkpointing": True, "noise_scheduler": "flowmatch",
                      "optimizer": "adamw8bit", "lr": 1e-4, "dtype": "bf16"},
            "model": {"name_or_path": "black-forest-labs/FLUX.1-dev", "is_flux": True,
                      "quantize": True, "low_vram": True},
            "sample": {"sampler": "flowmatch", "sample_every": STEPS + 1, "width": 512,
                       "height": 512, "prompts": [f"{trigger} portrait"], "sample_steps": 20},
        }]},
        "meta": {"name": name, "version": "1.0"},
    }
    path = f"/content/config_{job['id'][:8]}.yaml"
    with open(path, "w") as f:
        yaml.dump(config, f, sort_keys=False)
    return path, name

def upload_lora(lora_path, name):
    g = requests.post(f"{BACKEND_URL}/api/get-upload-url",
                      json={"fileName": f"{name}.safetensors", "fileType": "application/octet-stream"},
                      timeout=60)
    g.raise_for_status(); up = g.json()
    with open(lora_path, "rb") as f:
        requests.put(up["uploadURL"], data=f,
                     headers={"Content-Type": "application/octet-stream"}, timeout=1800).raise_for_status()
    return up["publicURL"]

def complete(job_id, lora_url=None, failed=False):
    body = {"failed": True} if failed else {"loraUrl": lora_url}
    requests.post(f"{BACKEND_URL}/worker/training/{job_id}/complete",
                  json=body, headers=HEADERS, timeout=60).raise_for_status()

def train_job(job):
    dataset_dir, trigger = prepare_dataset(job)
    cfg_path, name = write_config(job, dataset_dir, trigger)
    print(f"   training {name} ({len(job['imageUrls'])} images, trigger '{trigger}')")
    subprocess.run(["python", "run.py", cfg_path], cwd="/content/ai-toolkit", check=True)
    files = sorted(glob.glob(f"/content/output/{name}/*.safetensors"))
    if not files:
        raise RuntimeError("no safetensors produced")
    return upload_lora(files[-1], name)

## 4. Run the training worker loop (keep this running)

In [ ]:
seen = set()  # in-memory de-dupe (single worker)
print("training worker started; polling", BACKEND_URL)

while True:
    try:
        jr = requests.get(f"{BACKEND_URL}/worker/training-jobs", headers=HEADERS, timeout=60)
        jr.raise_for_status()
        jobs = [j for j in jr.json().get("jobs", []) if j["id"] not in seen]
        if not jobs:
            time.sleep(POLL_SECONDS); continue

        for job in jobs:
            seen.add(job["id"])
            print(f"-> training job {job['id']}")
            try:
                lora_url = train_job(job)
                complete(job["id"], lora_url=lora_url)
                print("   done:", lora_url)
            except Exception as e:
                print("   training failed:", e)
                try: complete(job["id"], failed=True)
                except Exception as e2: print("   (could not mark failed:", e2, ")")
    except KeyboardInterrupt:
        print("stopped"); break
    except Exception as e:
        print("poll error:", e); time.sleep(POLL_SECONDS * 2)